# Lab 03 — From a Real Hubble Diagram to a Cosmological Likelihood

**Course:** MSc Astrostatistics for Observational Cosmology  
**Format:** 4-hour hands-on practical  
**Status:** Formative + weekly checkpoint  
**Real data:** Pantheon+ / SH0ES  
**Covariance:** published Pantheon+ statistical covariance

> ## Scientific question
> **How do we turn real standardized SN Ia measurements into a likelihood for a cosmological parameter without confusing calibration with cosmology?**

Today we move from descriptive residuals to parameter inference.

We model the standardized apparent magnitude as

\[
m_{B,i}^{\rm corr}
=
\mu(z_i;\Omega_m,H_0)+M_B+\epsilon_i,
\]

where \(M_B\) is a nuisance absolute-magnitude calibration parameter.

For flat \(\Lambda\)CDM,

\[
H(z)=H_0\sqrt{\Omega_m(1+z)^3+1-\Omega_m}.
\]

The luminosity distance is

\[
D_L(z)=(1+z)c\int_0^z\frac{dz'}{H(z')},
\]

and

\[
\mu(z)=5\log_{10}\left(\frac{D_L}{\rm Mpc}\right)+25.
\]

## Critical identifiability point

SN apparent magnitudes alone do not separately determine \(H_0\) and \(M_B\). In this lab we fix

\[
H_0=70~{\rm km\,s^{-1}\,Mpc^{-1}}
\]

as a **convention**, and fit \(\Omega_m\) while profiling over \(M_B\).

The value of \(M_B\) therefore depends on the adopted \(H_0\) convention.

## 0. Five-line statement

**Estimand:**  
**Observed data:**  
**Generative assumptions:**  
**Uncertainty / selection:**  
**Validation / falsification test:**

In [ ]:
from pathlib import Path
import io, sys, time, requests, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.linalg import cho_factor, cho_solve
from scipy.interpolate import RegularGridInterpolator
from scipy.special import logsumexp
from astropy.cosmology import FlatLambdaCDM

SEED = 2026
rng = np.random.default_rng(SEED)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

## 1. Load the real Pantheon+ table and statistical covariance

In [ ]:
BASE = (
    "https://raw.githubusercontent.com/PantheonPlusSH0ES/"
    "DataRelease/main/Pantheon%2B_Data/4_DISTANCES_AND_COVAR/"
)
URL_DATA = BASE + "Pantheon%2BSH0ES.dat"
URL_STAT = BASE + "Pantheon%2BSH0ES_STATONLY.cov"
URL_FULL = BASE + "Pantheon%2BSH0ES_STAT%2BSYS.cov"

CACHE = Path("data_cache/pantheon_plus")
CACHE.mkdir(parents=True, exist_ok=True)

def download_if_needed(url, path, timeout=180):
    path = Path(path)
    if not path.exists():
        print("Downloading", path.name, "...")
        r = requests.get(url, timeout=timeout)
        r.raise_for_status()
        path.write_bytes(r.content)
    return path

def load_pantheon_table():
    p = download_if_needed(URL_DATA, CACHE/"Pantheon+SH0ES.dat")
    return pd.read_csv(p, sep=r"\s+")

def load_covariance(kind="STATONLY"):
    if kind.upper() == "STATONLY":
        url = URL_STAT
        path = CACHE/"Pantheon+SH0ES_STATONLY.cov"
    elif kind.upper() in {"FULL","STAT+SYS","STATSYS"}:
        url = URL_FULL
        path = CACHE/"Pantheon+SH0ES_STAT+SYS.cov"
    else:
        raise ValueError("kind must be STATONLY or FULL")

    p = download_if_needed(url, path)
    flat = np.loadtxt(p)
    n = int(flat[0])
    values = flat[1:]
    if values.size != n*n:
        raise ValueError(f"Expected {n*n} covariance entries, found {values.size}")
    return values.reshape(n, n)

sn = load_pantheon_table()
print("Pantheon+ rows:", len(sn))

In [ ]:
cov_stat = load_covariance("STATONLY")

print("Data length:", len(sn))
print("Covariance shape:", cov_stat.shape)

assert cov_stat.shape == (len(sn), len(sn))
assert np.allclose(cov_stat, cov_stat.T, atol=1e-12)
print("Covariance dimensions and symmetry: OK")

## 2. Define the teaching analysis sample

We use real non-calibrator SNe with

\[
z_{\rm HD}>0.01.
\]

This is a transparent **teaching sample**, not a claim to reproduce the exact Pantheon+ publication likelihood.

Crucially, we keep the corresponding submatrix of the published covariance in exactly the same row order.

In [ ]:
def teaching_mask(sn):
    """Real-data teaching sample: non-Cepheid calibrators with zHD > 0.01."""
    mask = (
        np.isfinite(sn["zHD"].to_numpy(float)) &
        np.isfinite(sn["m_b_corr"].to_numpy(float)) &
        (sn["zHD"].to_numpy(float) > 0.01) &
        (sn["IS_CALIBRATOR"].to_numpy(int) == 0)
    )
    return mask

def subset_data_cov(sn, cov, mask):
    idx = np.flatnonzero(mask)
    d = sn.iloc[idx].reset_index(drop=True).copy()
    c = cov[np.ix_(idx, idx)]
    return d, c, idx

def mu_flat_lcdm(z, omega_m, H0=70.0):
    cosmo = FlatLambdaCDM(H0=H0, Om0=omega_m, Tcmb0=2.725)
    return cosmo.distmod(np.asarray(z, float)).value

def coefficient_grid(data, cov, omega_grid, H0=70.0):
    """
    For model m_b_corr = mu(z; Omega_m,H0) + M_B,
    compute coefficients so that
    chi2(Omega_m, M_B) = a - 2 M_B b + M_B^2 d.
    """
    z = data["zHD"].to_numpy(float)
    m = data["m_b_corr"].to_numpy(float)
    one = np.ones(len(data))

    cf = cho_factor(cov, lower=True, check_finite=False)
    cinv_one = cho_solve(cf, one, check_finite=False)
    dcoef = one @ cinv_one

    a = np.empty(len(omega_grid))
    b = np.empty(len(omega_grid))

    for k, om in enumerate(omega_grid):
        y = m - mu_flat_lcdm(z, om, H0=H0)
        cinv_y = cho_solve(cf, y, check_finite=False)
        a[k] = y @ cinv_y
        b[k] = one @ cinv_y

    return a, b, float(dcoef)

def profile_mb(a, b, d):
    mb_hat = b/d
    chi2_prof = a - b*b/d
    return mb_hat, chi2_prof

def chi2_surface(a, b, d, mb_grid):
    mb = np.asarray(mb_grid, float)[None, :]
    return a[:,None] - 2*b[:,None]*mb + d*mb*mb

In [ ]:
mask = teaching_mask(sn)
data, Cstat, original_indices = subset_data_cov(sn, cov_stat, mask)

print("Teaching-sample rows:", len(data))
print("Redshift range:", data["zHD"].min(), "to", data["zHD"].max())
print("Covariance subset:", Cstat.shape)

### Exercise 1 — covariance sanity check

Calculate:

1. minimum and maximum diagonal variance;
2. median diagonal standard deviation;
3. fraction of non-zero off-diagonal elements;
4. the largest absolute correlation coefficient outside the diagonal.

Use

\[
R_{ij}=\frac{C_{ij}}{\sqrt{C_{ii}C_{jj}}}.
\]

**Question:** Why is a non-zero off-diagonal covariance scientifically plausible even before adding systematic errors?

In [ ]:
# TODO
diag = np.diag(Cstat)
# sigma = ...
# corr = ...

# Part I — Construct the likelihood

## 3. Multivariate Gaussian likelihood

Let

\[
\mathbf r(\Omega_m,M_B)
=
\mathbf m_B^{\rm corr}
-
\boldsymbol{\mu}(z;\Omega_m,H_0)
-
M_B\mathbf 1.
\]

Then, up to constants independent of the parameters,

\[
-2\ln\mathcal L
=
\chi^2
=
\mathbf r^T C^{-1}\mathbf r.
\]

Because \(M_B\) enters linearly, we do not need a 2-D optimiser.

Write

\[
\mathbf y(\Omega_m)
=
\mathbf m_B^{\rm corr}
-
\boldsymbol{\mu}(z;\Omega_m,H_0).
\]

Then

\[
\chi^2
=
a-2M_Bb+M_B^2d,
\]

with

\[
a=\mathbf y^TC^{-1}\mathbf y,
\qquad
b=\mathbf 1^TC^{-1}\mathbf y,
\qquad
d=\mathbf 1^TC^{-1}\mathbf 1.
\]

The profile value is therefore

\[
\widehat M_B(\Omega_m)=\frac{b}{d}.
\]

## 4. Build a profile-likelihood grid

Use

\[
0.10\le\Omega_m\le0.60.
\]

A grid is intentionally used before an optimiser because it allows you to **see the likelihood geometry**.

In [ ]:
OMEGA_GRID = np.linspace(0.10, 0.60, 81)
H0_CONVENTION = 70.0

t0 = time.time()
a_stat, b_stat, d_stat = coefficient_grid(
    data, Cstat, OMEGA_GRID, H0=H0_CONVENTION
)
mb_hat_stat, chi2_prof_stat = profile_mb(a_stat, b_stat, d_stat)

print(f"Grid calculation: {time.time()-t0:.1f} s")

i_best = np.argmin(chi2_prof_stat)
om_mle = OMEGA_GRID[i_best]
mb_mle = mb_hat_stat[i_best]
chi2_min = chi2_prof_stat[i_best]

print("Profile MLE Omega_m =", om_mle)
print("Profile MLE M_B =", mb_mle)
print("Minimum chi2 =", chi2_min)

## 5. Visualize the profile likelihood

For one fitted parameter, a common likelihood-based reference is

\[
\Delta\chi^2
=
\chi^2(\Omega_m)-\chi^2_{\min}.
\]

Do not automatically equate every \(\Delta\chi^2\) crossing with an exact frequentist confidence interval unless the regularity assumptions are satisfied.

In [ ]:
delta = chi2_prof_stat - chi2_min

fig, ax = plt.subplots(figsize=(7,4))
ax.plot(OMEGA_GRID, delta)
ax.axhline(1.0, ls="--", label=r"$\Delta\chi^2=1$")
ax.axvline(om_mle, ls=":", label=f"MLE = {om_mle:.3f}")
ax.set_xlabel(r"$\Omega_m$")
ax.set_ylabel(r"$\Delta\chi^2$")
ax.set_title("Pantheon+ statistical-covariance profile likelihood")
ax.legend()
plt.show()

### Exercise 2 — numerical interval

Find the grid points satisfying

\[
\Delta\chi^2\le 1.
\]

Report the resulting approximate interval.

Then refine the grid locally around the minimum and check whether the reported interval changes appreciably.

In [ ]:
# TODO:
# inside = OMEGA_GRID[delta <= 1]
# print(...)

# Part II — What happened to \(M_B\)?

## 6. Nuisance parameter profile

Plot

\[
\widehat M_B(\Omega_m)
\]

across the grid.

Then repeat the coefficient calculation using

\[
H_0=73.
\]

### Prediction before running

Will the best-fitting \(\Omega_m\) shift strongly?

Will \(\widehat M_B\) shift?

Write your prediction first.

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(OMEGA_GRID, mb_hat_stat)
ax.set_xlabel(r"$\Omega_m$")
ax.set_ylabel(r"profile $\widehat{M}_B$")
ax.set_title(r"Nuisance calibration profile ($H_0=70$ convention)")
plt.show()

# TODO: repeat with H0=73 and compare.

### Key interpretation

Because

\[
D_L\propto H_0^{-1},
\]

changing \(H_0\) changes the distance-modulus zero point. The nuisance magnitude absorbs that change.

This is a concrete example of **parameter non-identifiability / degeneracy**, not a numerical inconvenience.

# Part III — Residual diagnostics at the MLE

## 7. Inspect what the fitted model fails to explain

Calculate residuals at

\[
(\widehat\Omega_m,\widehat M_B).
\]

Plot:

1. residual versus redshift;
2. residual histogram;
3. standardized residual using \(\sqrt{C_{ii}}\) **only as a diagnostic visualization**.

Do not confuse standardized diagonal residuals with the full correlated likelihood.

In [ ]:
mu_best = mu_flat_lcdm(data["zHD"], om_mle, H0=H0_CONVENTION)
resid_best = data["m_b_corr"].to_numpy(float) - mu_best - mb_mle

fig, ax = plt.subplots(figsize=(8,4))
ax.axhline(0, lw=1)
ax.scatter(data["zHD"], resid_best, s=8, alpha=0.55)
ax.set_xscale("log")
ax.set_xlabel(r"$z_{\rm HD}$")
ax.set_ylabel("Magnitude residual [mag]")
ax.set_title("Residuals at profile maximum")
plt.show()

# TODO: histogram and diagonal-standardized residual plot.

# 8. In-class checkpoint

Show:

1. covariance sanity-check results;
2. \(\Delta\chi^2(\Omega_m)\);
3. approximate likelihood interval;
4. \(\widehat M_B(\Omega_m)\);
5. the \(H_0=70\) versus \(H_0=73\) comparison;
6. residual diagnostic.

### Exit question

In 100–150 words:

> Why is “fit \(H_0\) from SN apparent magnitudes” ill-posed unless an absolute calibration is supplied?

# 9. Bridge to Lab 04

Today you found the maximum of

\[
\mathcal L(D\mid\Omega_m,M_B).
\]

Next week you will construct

\[
p(\Omega_m,M_B\mid D)
\propto
\mathcal L(D\mid\Omega_m,M_B)
p(\Omega_m,M_B),
\]

normalize it, marginalize it, and test prior sensitivity.